# LSTM Complex Concatenate

Notebook version of the complex concatenation LSTM pipeline with GloVe embeddings and class weighting.

This notebook was converted from the corresponding `.py` script and keeps the same overall logic and output paths.

In [1]:
import json
import pickle
import re
import zipfile
import nltk
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight


from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional, Dropout, Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical


TRAIN_PATH = "twitter_training (1).csv"
VAL_PATH = "twitter_validation (1).csv"
OUTPUT_DIR = Path("outputs/lstm_complex_concatenate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GLOVE_ZIP_PATH = Path("glove.6B.zip")
GLOVE_TXT_PATH = Path("glove.6B.100d.txt")
GLOVE_URL = "https://nlp.stanford.edu/data/glove.6B.zip"


nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        header=None,
        names=["tweet_id", "entity", "sentiment", "tweet_content"],
    )
    df["tweet_content"] = df["tweet_content"].fillna("").astype(str)
    df["sentiment"] = df["sentiment"].astype(str)
    return df


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def concatenate_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df.groupby(["tweet_id", "entity", "sentiment"], as_index=False)["tweet_content"]
        .apply(lambda x: " ".join(x.astype(str)))
    )
    return grouped.reset_index(drop=True)


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = concatenate_duplicates(df).copy()
    df["cleaned_tweet_content"] = df["tweet_content"].apply(clean_text)
    df["tokenized_tweet_content"] = df["cleaned_tweet_content"].apply(lambda x: x.split())
    df["filtered_tweet_content"] = df["tokenized_tweet_content"].apply(
        lambda tokens: [word for word in tokens if word not in STOP_WORDS]
    )
    df["lemmatized_tweet_content"] = df["filtered_tweet_content"].apply(
        lambda tokens: [LEMMATIZER.lemmatize(word) for word in tokens]
    )
    df["lemmatized_text_string"] = df["lemmatized_tweet_content"].apply(" ".join)
    return df


def prepare_features(train_df: pd.DataFrame, val_df: pd.DataFrame):
    tokenizer = Tokenizer(num_words=10000, oov_token="<unk>")
    tokenizer.fit_on_texts(train_df["lemmatized_text_string"])

    train_sequences = tokenizer.texts_to_sequences(train_df["lemmatized_text_string"])
    val_sequences = tokenizer.texts_to_sequences(val_df["lemmatized_text_string"])

    sequence_lengths = [len(seq) for seq in train_sequences if len(seq) > 0]
    max_sequence_length = int(np.percentile(sequence_lengths, 95)) if sequence_lengths else 150
    max_sequence_length = max(max_sequence_length, 20)

    X_train = pad_sequences(train_sequences, maxlen=max_sequence_length, padding="post", truncating="post")
    X_val = pad_sequences(val_sequences, maxlen=max_sequence_length, padding="post", truncating="post")
    return tokenizer, X_train, X_val, max_sequence_length


def encode_labels(train_df: pd.DataFrame, val_df: pd.DataFrame):
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(train_df["sentiment"])
    y_val_encoded = label_encoder.transform(val_df["sentiment"])
    y_train = to_categorical(y_train_encoded, num_classes=len(label_encoder.classes_))
    y_val = to_categorical(y_val_encoded, num_classes=len(label_encoder.classes_))
    return label_encoder, y_train, y_val, y_train_encoded, y_val_encoded


def ensure_glove() -> None:
    if GLOVE_TXT_PATH.exists():
        return
    if not GLOVE_ZIP_PATH.exists():
        print("Downloading GloVe embeddings...")
        urllib.request.urlretrieve(GLOVE_URL, GLOVE_ZIP_PATH)
    print("Extracting GloVe embeddings...")
    with zipfile.ZipFile(GLOVE_ZIP_PATH, "r") as zf:
        zf.extract("glove.6B.100d.txt", path=".")


def load_glove_embeddings() -> dict:
    ensure_glove()
    embeddings_index = {}
    with open(GLOVE_TXT_PATH, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype="float32")
            embeddings_index[word] = coefs
    return embeddings_index


def build_embedding_matrix(tokenizer: Tokenizer, embedding_dim: int = 100) -> np.ndarray:
    embeddings_index = load_glove_embeddings()
    vocab_size = min(len(tokenizer.word_index) + 1, 10000)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    for word, idx in tokenizer.word_index.items():
        if idx >= vocab_size:
            continue
        vector = embeddings_index.get(word)
        if vector is not None:
            embedding_matrix[idx] = vector

    return embedding_matrix


def build_model(vocab_size: int, embedding_matrix: np.ndarray, max_sequence_length: int, num_classes: int) -> Sequential:
    model = Sequential()
    model.add(
        Embedding(
            input_dim=vocab_size,
            output_dim=100,
            weights=[embedding_matrix],
            input_shape=(max_sequence_length,),
            trainable=False,
        )
    )
    model.add(Bidirectional(LSTM(units=128, return_sequences=False)))
    model.add(Dropout(0.3))
    model.add(Dense(units=num_classes, activation="softmax"))
    model.compile(optimizer=Adam(), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def plot_history(history) -> None:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title("Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_history.png", dpi=200, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, class_names: np.ndarray) -> None:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()


def main() -> None:
    train_df = preprocess_dataframe(load_dataset(TRAIN_PATH))
    val_df = preprocess_dataframe(load_dataset(VAL_PATH))

    tokenizer, X_train, X_val, max_sequence_length = prepare_features(train_df, val_df)
    label_encoder, y_train, y_val, y_train_encoded, y_val_encoded = encode_labels(train_df, val_df)

    class_weights = class_weight.compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train_encoded),
        y=y_train_encoded,
    )
    class_weights_dict = dict(enumerate(class_weights))

    embedding_matrix = build_embedding_matrix(tokenizer, embedding_dim=100)
    vocab_size = embedding_matrix.shape[0]
    model = build_model(vocab_size, embedding_matrix, max_sequence_length, len(label_encoder.classes_))

    early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

    history = model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping],
        class_weight=class_weights_dict,
        verbose=1,
    )

    loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
    y_pred_probs = model.predict(X_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    print(f"Validation Loss: {loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val_encoded, y_pred, target_names=label_encoder.classes_))

    plot_history(history)
    plot_confusion_matrix(y_val_encoded, y_pred, label_encoder.classes_)

    model.save(OUTPUT_DIR / "lstm_complex_concatenate.keras")
    with open(OUTPUT_DIR / "tokenizer.pkl", "wb") as f:
        pickle.dump(tokenizer, f)
    with open(OUTPUT_DIR / "label_encoder.pkl", "wb") as f:
        pickle.dump(label_encoder, f)
    with open(OUTPUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "train_rows_after_concatenation": int(len(train_df)),
                "validation_rows_after_concatenation": int(len(val_df)),
                "vocab_size": int(vocab_size),
                "max_sequence_length": int(max_sequence_length),
                "classes": label_encoder.classes_.tolist(),
                "class_weights": {label_encoder.classes_[k]: float(v) for k, v in class_weights_dict.items()},
                "validation_loss": float(loss),
                "validation_accuracy": float(accuracy),
            },
            f,
            indent=2,
        )



In [2]:
main()

Extracting GloVe embeddings...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 145s 359ms/step - accuracy: 0.4608 - loss: 1.2275 - val_accuracy: 0.5000 - val_loss: 1.1948
Epoch 2/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 139s 358ms/step - accuracy: 0.5244 - loss: 1.1302 - val_accuracy: 0.5160 - val_loss: 1.1506
Epoch 3/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 140s 361ms/step - accuracy: 0.5468 - loss: 1.0778 - val_accuracy: 0.5380 - val_loss: 1.1124
Epoch 4/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 138s 354ms/step - accuracy: 0.5750 - loss: 1.0294 - val_accuracy: 0.5630 - val_loss: 1.0843
Epoch 5/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 139s 359ms/step - accuracy: 0.6060 - loss: 0.9699 - val_accuracy: 0.6040 - val_loss: 1.0479
Epoch 6/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 141s 363ms/step - accuracy: 0.6235 - loss: 0.9162 - val_accuracy: 0.5720 - val_loss: 1.0388
Epoch 7/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 141s 363ms/step - accuracy: 0.6508 - loss: 0.8607 - val_accuracy: 0.6030 - val_loss: 1.0181
Epoch 8/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 142s 364ms/step - accuracy: 0.6782 -